In [1]:
import io
import operator
import os
import sys
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from research import ResearchAssistant, ResearchSchema

_ = load_dotenv()
llm = init_chat_model("deepseek-chat", temperature=0)

In [2]:
#  1. BASIC LINEAR PIPELINE 
# Retrieve -> Format -> Generate answer from a research assistant


class ResearchState(TypedDict):
    question: str
    docs: list
    context: str
    result: str


assistant = ResearchAssistant()


def retrieve_docs(state: ResearchState) -> ResearchState:
    state["docs"] = assistant.get_retriever().invoke(state["question"])
    return state


def format_context(state: ResearchState) -> ResearchState:
    state["context"] = assistant._format_docs_for_context(state["docs"])
    return state


def generate_answer(state: ResearchState) -> ResearchState:
    chain = assistant._build_prompt() | assistant.llm.with_structured_output(
        ResearchSchema
    )
    state["result"] = chain.invoke(
        {
            "context": state["context"],
            "question": state["question"],
            "history": [],
        }
    )
    return state


graph = StateGraph(ResearchState)
graph.add_node("retrieve", retrieve_docs)
graph.add_node("format", format_context)
graph.add_node("generate", generate_answer)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "format")
graph.add_edge("format", "generate")
graph.add_edge("generate", END)

runnable = graph.compile()

result = runnable.invoke(
    {"question": "What is the bias-variance tradeoff in supervised learning?"}
)
assistant.print_response(result["result"], question=result["question"])

Q: What is the bias-variance tradeoff in supervised learning?
[HIGH] The bias-variance tradeoff is a central concept in statistical learning that describes the relationship between model complexity and prediction error. 

**Bias** is the error due to overly simplistic model assumptions that cause the model to miss relevant patterns in the data. **Variance** is the error due to sensitivity to fluctuations in the training data, meaning the model changes significantly when trained on different datasets.

As model complexity increases:
- **Bias decreases** (the model becomes flexible enough to capture patterns)
- **Variance increases** (the model becomes more sensitive to noise in the training data)

The optimal model minimizes the total expected error, which is calculated as: **bias² + variance + irreducible noise**.

This tradeoff explains two common problems:
- **Overfitting**: When a model is too complex, it learns noise in the training data and performs poorly on new data (low bias, h

In [3]:
# 2. STATE REDUCERS AND ACCUMULATORS
# Demonstrating how annotated fields accumulate across nodes


class AccumulatingState(TypedDict):
    messages: Annotated[list[str], operator.add]
    count: Annotated[int, operator.add]


def step_one(state: AccumulatingState) -> dict:
    return {"messages": ["Preprocessing complete"], "count": 1}


def step_two(state: AccumulatingState) -> dict:
    return {"messages": ["Model training complete"], "count": 1}


graph = StateGraph(AccumulatingState)
graph.add_node("step_one", step_one)
graph.add_node("step_two", step_two)
graph.set_entry_point("step_one")
graph.add_edge("step_one", "step_two")
graph.set_finish_point("step_two")

app = graph.compile()
result = app.invoke({"messages": ["Pipeline initialized"], "count": 0})
print(f"Messages: {result['messages']}, Count: {result['count']}")

Messages: ['Pipeline initialized', 'Preprocessing complete', 'Model training complete'], Count: 2


In [4]:
# 3. MULTI-NODE ANALYSIS PIPELINE
# Analyze -> Enhance -> Finalize a statistical concept


class MultiStepState(TypedDict):
    input: str
    analyzed: str
    enhanced: str
    final: str


def analyze_node(state: MultiStepState) -> dict:
    response = llm.invoke(
        [
            HumanMessage(
                content=f"In one sentence, summarize the core statistical idea behind: {state['input']}"
            )
        ]
    )
    return {"analyzed": response.content}


def enhance_node(state: MultiStepState) -> dict:
    response = llm.invoke(
        [
            HumanMessage(
                content=(
                    f"Expand on this statistical summary with assumptions, use-cases, and limitations:\n\n"
                    f"{state['analyzed']}"
                )
            )
        ]
    )
    return {"enhanced": response.content}


def finalize_node(state: MultiStepState) -> dict:
    response = llm.invoke(
        [
            HumanMessage(
                content=f"Condense the following into a two-sentence practitioner summary:\n\n{state['enhanced']}"
            )
        ]
    )
    return {"final": response.content}


graph = StateGraph(MultiStepState)
graph.add_node("analyze", analyze_node)
graph.add_node("enhance", enhance_node)
graph.add_node("finalize", finalize_node)
graph.set_entry_point("analyze")
graph.add_edge("analyze", "enhance")
graph.add_edge("enhance", "finalize")
graph.set_finish_point("finalize")

app = graph.compile()
result = app.invoke({"input": "Gradient Boosting"})
print(
    f"Analyzed: {result['analyzed']}\n\nEnhanced: {result['enhanced']}\n\nFinal: {result['final']}"
)

Analyzed: Gradient boosting sequentially builds an ensemble of weak learners, where each new model is trained to predict the **negative gradient of the loss function** with respect to the current ensemble's predictions, effectively performing gradient descent in function space.

Enhanced: Here is a detailed expansion of the provided statistical summary, breaking down the core mechanism, the underlying assumptions, practical use-cases, and critical limitations.

### Core Mechanism (Expanding the Summary)

The summary correctly identifies the two key pillars of Gradient Boosting: **sequential ensemble building** and **gradient descent in function space**. Let's unpack these.

1.  **Sequential Ensemble of Weak Learners:** Unlike bagging (e.g., Random Forest) where models are built independently in parallel, Gradient Boosting builds models one after another. Each new model (`h_m(x)`) is specifically designed to correct the errors of the combined ensemble built so far (`F_{m-1}(x)`). The "w

In [5]:
# 4. CONDITIONAL ROUTING BY QUERY TYPE
# Route ML queries to the appropriate handler based on classification


class RouterState(TypedDict):
    query: str
    query_type: str
    response: str


def classify_query(state: RouterState) -> dict:
    response = llm.invoke(
        f"Classify the following ML/stats query as exactly one of: 'concept', 'implementation', 'comparison'.\n"
        f"Reply with just the word.\n\nQuery: {state['query']}"
    )
    return {"query_type": response.content.lower().strip()}


def handle_concept(state: RouterState) -> dict:
    response = llm.invoke(
        f"Explain this ML/stats concept clearly for a data scientist:\n\n{state['query']}"
    )
    return {"response": f"[Concept] {response.content}"}


def handle_implementation(state: RouterState) -> dict:
    response = llm.invoke(
        f"Provide a concise Python implementation with sklearn or numpy for:\n\n{state['query']}"
    )
    return {"response": f"[Implementation] {response.content}"}


def handle_comparison(state: RouterState) -> dict:
    response = llm.invoke(
        f"Compare the approaches mentioned, covering tradeoffs, assumptions, and when to prefer each:\n\n{state['query']}"
    )
    return {"response": f"[Comparison] {response.content}"}


def route_by_type(state: RouterState) -> str:
    qt = state["query_type"]
    if "concept" in qt:
        return "concept"
    elif "implementation" in qt:
        return "implementation"
    else:
        return "comparison"


graph = StateGraph(RouterState)
graph.add_node("classify", classify_query)
graph.add_node("handle_concept", handle_concept)
graph.add_node("handle_implementation", handle_implementation)
graph.add_node("handle_comparison", handle_comparison)
graph.set_entry_point("classify")
graph.add_conditional_edges(
    "classify",
    route_by_type,
    {
        "concept": "handle_concept",
        "implementation": "handle_implementation",
        "comparison": "handle_comparison",
    },
)
graph.set_finish_point("handle_concept")
graph.set_finish_point("handle_implementation")
graph.set_finish_point("handle_comparison")

app = graph.compile()

queries = [
    "What is regularization in linear regression?",
    "How do I implement k-means clustering in sklearn?",
    "Random forest vs gradient boosting: which should I choose?",
]
for query in queries:
    result = app.invoke({"query": query})
    print(
        f"Query: {query}\nType: {result['query_type']}\nResponse: {result['response']}\n{'-' * 60}"
    )

Query: What is regularization in linear regression?
Type: concept
Response: [Concept] Here’s a clear, data-scientist-friendly explanation of regularization in linear regression.

---

### The Core Problem: Overfitting & Unstable Coefficients

Standard linear regression (Ordinary Least Squares, OLS) finds coefficients that minimize the **sum of squared errors** on the training data. This works well when:
- You have many more observations than features.
- Features are not highly correlated.

**The problem:** OLS can become unstable or overfit when:
- **You have many features** (especially more than observations, p > n).
- **Features are highly correlated** (multicollinearity). The model can assign huge positive coefficients to one feature and huge negative coefficients to a correlated one, canceling each other out on training data but failing on new data.

**Regularization** is a technique that **penalizes large coefficients** to prevent this instability and overfitting.

---

### The In

In [6]:
# 5. SENTIMENT-AWARE CONVERSATION
# Analyze message sentiment, then craft a contextually appropriate reply


class ConversationState(TypedDict):
    messages: Annotated[list, operator.add]
    sentiment: Literal["positive", "negative", "neutral"]
    response_count: int


def analyze_sentiment(state: ConversationState) -> dict:
    response = llm.invoke(
        [
            SystemMessage(
                content="Classify the sentiment of the message as exactly one of: positive, negative, neutral. Reply with just the word."
            ),
            HumanMessage(content=state["messages"][-1]),
        ]
    )
    return {"sentiment": response.content.lower().strip()}


def generate_response(state: ConversationState) -> dict:
    system_prompts = {
        "positive": "The user is engaged and enthusiastic. Match their energy and build on it.",
        "negative": "The user is frustrated. Acknowledge the difficulty and offer practical help.",
        "neutral": "Respond clearly, helpfully, and concisely.",
    }
    response = llm.invoke(
        [
            SystemMessage(
                content=system_prompts.get(
                    state["sentiment"], system_prompts["neutral"]
                )
            ),
            HumanMessage(content=state["messages"][-1]),
        ]
    )
    return {"messages": [f"AI: {response.content}"], "response_count": 1}


graph = StateGraph(ConversationState)
graph.add_node("analyze_sentiment", analyze_sentiment)
graph.add_node("generate_response", generate_response)
graph.set_entry_point("analyze_sentiment")
graph.add_edge("analyze_sentiment", "generate_response")
graph.set_finish_point("generate_response")

app = graph.compile()

test_messages = [
    "My random forest model just hit 98% accuracy on the validation set!",
    "The model keeps overfitting no matter what regularization I try.",
    "Cross-validation splits the data into k folds for evaluation.",
]
for msg in test_messages:
    result = app.invoke({"messages": [f"Human: {msg}"]})
    print(
        f"Input: {msg}\nSentiment: {result['sentiment']}\nResponse: {result['messages'][-1]}\n{'--' * 40}"
    )

Input: My random forest model just hit 98% accuracy on the validation set!
Sentiment: positive
Response: AI: That's fantastic — 98% accuracy is a solid milestone! 🎉 You've clearly put in good work on feature engineering, hyperparameter tuning, or data preprocessing to get there.

What's next on your radar? Are you thinking about:
- Testing on a holdout test set to check for overfitting?
- Diving into feature importance to see which variables are driving those predictions?
- Or maybe deploying the model into production?

Either way, that's a great result — you should be proud!
--------------------------------------------------------------------------------
Input: The model keeps overfitting no matter what regularization I try.
Sentiment: negative
Response: AI: That sounds incredibly frustrating. You've tried regularization, and it's still not working—that's a tough spot to be in.

Let's break this down. Overfitting usually means the model is memorizing the training data instead of learn

In [7]:
# 6. ITERATIVE QUALITY IMPROVEMENT LOOP
# Generate ML content, score it, improve until threshold or max iterations

MAX_ITERATIONS = 3
QUALITY_THRESHOLD = 7

llm_creative = init_chat_model("deepseek-chat", temperature=0.7, max_tokens=2000)


class QualityState(BaseModel):
    content: str
    quality_score: int = 0
    feedback: str = ""
    final_content: str = ""
    iteration: int = 0


class RateScore(BaseModel):
    score: int = Field(description="Quality score from 1 to 10", ge=1, le=10)


def evaluate_quality(state: QualityState) -> Command:
    result = llm_creative.with_structured_output(RateScore).invoke(
        f"Rate this ML explanation from 1-10 on accuracy, clarity, and practical relevance.\n\nContent:\n{state.content}"
    )
    approved = result.score >= QUALITY_THRESHOLD or state.iteration >= MAX_ITERATIONS
    return Command(
        update={"quality_score": result.score},
        goto="finalize" if approved else "improve",
    )


def improve_content(state: QualityState) -> Command:
    response = llm_creative.invoke(
        f"Rewrite the following ML explanation to be more accurate, concrete, and useful for practitioners:\n\n{state.content}"
    )
    return Command(
        update={"content": response.content, "iteration": state.iteration + 1},
        goto="evaluate",
    )


def finalize_content(state: QualityState) -> Command:
    return Command(
        update={
            "final_content": state.content,
            "feedback": f"Approved after {state.iteration} iteration(s) with score {state.quality_score}/10",
        },
        goto=END,
    )


graph = StateGraph(QualityState)
graph.add_node("evaluate", evaluate_quality)
graph.add_node("improve", improve_content)
graph.add_node("finalize", finalize_content)
graph.add_edge(START, "evaluate")

app = graph.compile()

initial = QualityState(
    content="Decision trees split data randomly until they find patterns."
)
result = app.invoke(initial)
print(result["feedback"])
print(result["final_content"])

Approved after 1 iteration(s) with score 9/10
Here’s a more accurate, concrete, and practitioner-focused rewrite:

**Original:**  
"Decision trees split data randomly until they find patterns."

**Rewritten:**  
"Decision trees partition the feature space by recursively applying a series of deterministic, rule-based splits. At each node, the algorithm selects the split (feature and threshold) that maximizes a chosen criterion, such as Gini impurity or information gain, to best separate the target classes or reduce variance. Splits are not random—they are greedy optimizations based on the training data. The process continues until a stopping condition is met (e.g., maximum depth, minimum samples per leaf, or no further gain), which helps control overfitting. Practitioners should tune these hyperparameters and consider pruning or ensemble methods (e.g., random forests) to improve generalization."


In [8]:
# 7. MULTI-DIMENSIONAL TASK ROUTER
# Route ML tasks by urgency and complexity simultaneously


class TaskState(BaseModel):
    task: str
    urgency: Literal["urgent", "normal"] = "normal"
    complexity: Literal["complex", "simple"] = "simple"
    handler: str = ""
    result: str = ""


class UrgencySchema(BaseModel):
    urgency: Literal["urgent", "normal"] = Field(
        description="Whether the task requires immediate attention"
    )


class ComplexitySchema(BaseModel):
    complexity: Literal["complex", "simple"] = Field(
        description="Whether the task requires deep ML expertise"
    )


def analyze_task(state: TaskState) -> Command:
    urgency = (
        llm.with_structured_output(UrgencySchema)
        .invoke(
            f"Does this ML task require immediate attention, or can it be scheduled?\nTask: {state.task}"
        )
        .urgency
    )
    complexity = (
        llm.with_structured_output(ComplexitySchema)
        .invoke(
            f"Does this ML task require deep expertise and research, or is it straightforward?\nTask: {state.task}"
        )
        .complexity
    )
    return Command(
        update={"urgency": urgency, "complexity": complexity},
        goto=f"{urgency}_{complexity}",
    )


def urgent_complex_handler(state: TaskState) -> Command:
    return Command(
        update={
            "handler": "Senior ML Engineer",
            "result": "Escalated for immediate expert attention",
        },
        goto=END,
    )


def urgent_simple_handler(state: TaskState) -> Command:
    return Command(
        update={
            "handler": "On-Call Engineer",
            "result": "Handled immediately by on-call",
        },
        goto=END,
    )


def normal_complex_handler(state: TaskState) -> Command:
    return Command(
        update={
            "handler": "ML Specialist",
            "result": "Assigned to specialist for thorough investigation",
        },
        goto=END,
    )


def normal_simple_handler(state: TaskState) -> Command:
    return Command(
        update={"handler": "Standard Queue", "result": "Added to the sprint backlog"},
        goto=END,
    )


graph = StateGraph(TaskState)
graph.add_node("analyze", analyze_task)
graph.add_node("urgent_complex", urgent_complex_handler)
graph.add_node("urgent_simple", urgent_simple_handler)
graph.add_node("normal_complex", normal_complex_handler)
graph.add_node("normal_simple", normal_simple_handler)
graph.add_edge(START, "analyze")

app = graph.compile()

tasks = [
    "Production model is returning NaN predictions for 30% of requests.",
    "Retrain the churn model with this month's data.",
    "Design a new architecture for real-time anomaly detection at scale.",
    "Update the feature engineering docstring for the tenure variable.",
]
for task in tasks:
    result = app.invoke({"task": task})
    print(
        f"Task: {task}\nUrgency: {result['urgency']} | Complexity: {result['complexity']}"
    )
    print(f"Handler: {result['handler']}\nResult: {result['result']}\n{'-' * 60}")

Task: Production model is returning NaN predictions for 30% of requests.
Urgency: urgent | Complexity: complex
Handler: Senior ML Engineer
Result: Escalated for immediate expert attention
------------------------------------------------------------
Task: Retrain the churn model with this month's data.
Urgency: normal | Complexity: simple
Handler: Standard Queue
Result: Added to the sprint backlog
------------------------------------------------------------
Task: Design a new architecture for real-time anomaly detection at scale.
Urgency: urgent | Complexity: complex
Handler: Senior ML Engineer
Result: Escalated for immediate expert attention
------------------------------------------------------------
Task: Update the feature engineering docstring for the tenure variable.
Urgency: normal | Complexity: simple
Handler: Standard Queue
Result: Added to the sprint backlog
------------------------------------------------------------


In [9]:
# 8. SELF-CORRECTING CODE WRITER
# Generate, validate, and auto-fix Python code for ML/stats tasks


class CodeGenState(BaseModel):
    task: str
    code: str = ""
    errors: Annotated[list[str], operator.add] = Field(default_factory=list)
    iteration: int = 0
    max_iterations: int = 3
    success: bool = False


class CodeResponseSchema(BaseModel):
    code: str = Field(
        description="Executable Python code only. No markdown fences, no explanations."
    )


def generate_code(state: CodeGenState) -> dict:
    if state.iteration == 0:
        prompt = (
            f"Write clean, correct Python code for the following task. "
            f"Return only the executable code, no markdown:\n\n{state.task}"
        )
    else:
        prompt = (
            f"Fix the following Python code. Return only the corrected executable code.\n\n"
            f"Code:\n{state.code}\n\nErrors:\n{chr(10).join(state.errors)}"
        )
    response = llm.with_structured_output(CodeResponseSchema).invoke(prompt)
    return {"code": response.code, "iteration": state.iteration + 1}


def validate_code(state: CodeGenState) -> Command:
    if state.iteration >= state.max_iterations:
        return Command(update={"success": False}, goto=END)

    try:
        compiled = compile(state.code, "<string>", "exec")
    except Exception as e:
        return Command(
            update={
                "errors": [f"SyntaxError: {type(e).__name__}: {e}"],
                "success": False,
            },
            goto="generate_code",
        )

    old_stdout, sys.stdout = sys.stdout, io.StringIO()
    try:
        exec(compiled, {})
        return Command(update={"errors": [], "success": True}, goto=END)
    except Exception as e:
        return Command(
            update={
                "errors": [f"RuntimeError: {type(e).__name__}: {e}"],
                "success": False,
            },
            goto="generate_code",
        )
    finally:
        sys.stdout = old_stdout


graph = StateGraph(CodeGenState)
graph.add_node("generate_code", generate_code)
graph.add_node("validate_code", validate_code)
graph.add_edge(START, "generate_code")
graph.add_edge("generate_code", "validate_code")

app = graph.compile()

task = (
    "A function `compute_confusion_matrix(y_true, y_pred)` that returns a 2x2 confusion matrix "
    "as a nested list for binary classification. Handle empty inputs, mismatched lengths, and "
    "inputs with only one class. Do not use sklearn."
)
result = app.invoke(CodeGenState(task=task))
print(
    f"Iterations: {result['iteration']} | Success: {result['success']}\n{result['code']}"
)

Iterations: 1 | Success: True
def compute_confusion_matrix(y_true, y_pred):
    if not y_true or not y_pred:
        return [[0, 0], [0, 0]]
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have the same length")
    classes = set(y_true) | set(y_pred)
    if len(classes) == 1:
        cls = next(iter(classes))
        if cls == 1:
            tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
            fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
            fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
            tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
            return [[tn, fp], [fn, tp]]
        else:
            tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
            fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
            fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
            tp = sum(1 for t, p in zip

In [10]:
# 9. ITERATIVE DEEP RESEARCH WORKFLOW
# Research a topic progressively, generating follow-up questions each round


class DeepResearchState(BaseModel):
    topic: str
    findings: Annotated[list[str], operator.add] = Field(default_factory=list)
    questions: list[str] = Field(default_factory=list)
    iteration: int = 0
    max_depth: int = 2
    summary: str = ""


def research(state: DeepResearchState) -> dict:
    if state.iteration == 0:
        query = f"Give 3 key technical facts about {state.topic}, including assumptions and known limitations."
    else:
        query = (
            f"Previous finding:\n{state.findings[-1]}\n\n"
            f"Go deeper on: {state.questions[-1]}\n"
            f"Focus on practical implications for ML practitioners."
        )
    response = llm.invoke(query)
    return {"findings": [response.content]}


def generate_questions(state: DeepResearchState) -> Command:
    response = llm.invoke(
        f"Given this ML/stats finding:\n{state.findings[-1]}\n\n"
        "What is the single most important follow-up question a practitioner should investigate? "
        "Reply with just the question."
    )
    new_iteration = state.iteration + 1
    goto = "synthesize" if new_iteration >= state.max_depth else "research"
    return Command(
        update={"questions": [response.content.strip()], "iteration": new_iteration},
        goto=goto,
    )


def synthesize(state: DeepResearchState) -> dict:
    all_findings = "\n\n--\n\n".join(state.findings)
    response = llm.invoke(
        f"Synthesize these research findings into a structured summary for a data scientist. "
        f"Cover key concepts, practical takeaways, and open questions:\n\n{all_findings}"
    )
    return {"summary": response.content}


graph = StateGraph(DeepResearchState)
graph.add_node("research", research)
graph.add_node("generate_questions", generate_questions)
graph.add_node("synthesize", synthesize)
graph.add_edge(START, "research")
graph.add_edge("research", "generate_questions")
graph.add_edge("synthesize", END)

app = graph.compile()

result = app.invoke(
    DeepResearchState(
        topic="Bayesian optimization for hyperparameter tuning", max_depth=2
    )
)
print(
    f"Topic: {result['topic']}\nDepth: {result['iteration']}\nFindings: {len(result['findings'])}"
)
print(f"\nSummary:\n{result['summary']}")

Topic: Bayesian optimization for hyperparameter tuning
Depth: 2
Findings: 2

Summary:
# Bayesian Optimization for Hyperparameter Tuning: Structured Summary for Data Scientists

## Core Concepts

### 1. Probabilistic Surrogate Modeling
- **Primary mechanism**: Gaussian Process (GP) builds a probabilistic approximation of the objective function $f(x)$ mapping hyperparameters → model performance
- **Output**: Posterior distribution providing both mean prediction (expected performance) and variance (uncertainty) for any unobserved configuration
- **Key assumption**: Objective function must be **smooth and continuous** (enforced via kernel choice like RBF or Matern)
- **Computational cost**: $O(n^3)$ for covariance matrix inversion, limiting scalability to ~20 hyperparameters and moderate evaluation budgets

### 2. Acquisition Function for Exploration-Exploitation
- **Expected Improvement (EI)**: $EI(x) = \mathbb{E}[\max(0, f(x) - f(x^+))]$ balances high-mean regions (exploitation) with hig

In [11]:
# 10. HUMAN-IN-THE-LOOP APPROVAL
# Draft a technical document, pause for human review, revise on feedback

checkpointer = InMemorySaver()


class ApprovalState(TypedDict):
    request: str
    draft: str
    final: str


def create_draft(state: ApprovalState) -> dict:
    response = llm.invoke(
        f"Write a concise technical explanation suitable for a data science blog post:\n\n{state['request']}"
    )
    return {"draft": response.content}


def approval_node(state: ApprovalState) -> Command:
    human_input = interrupt(
        {
            "message": "Review the draft. Approve or provide revision feedback.",
            "draft": state["draft"],
        }
    )
    if human_input.get("approved", False):
        return Command(goto="finalize", update={"final": state["draft"]})
    revised = llm.invoke(
        f"Revise the following technical explanation based on the feedback provided.\n\n"
        f"Draft:\n{state['draft']}\n\nFeedback:\n{human_input.get('feedback', '')}"
    )
    return Command(goto="finalize", update={"final": revised.content})


def finalize(state: ApprovalState) -> dict:
    return {}


graph = StateGraph(ApprovalState)
graph.add_node("draft", create_draft)
graph.add_node("approval", approval_node)
graph.add_node("finalize", finalize)
graph.set_entry_point("draft")
graph.add_edge("draft", "approval")
graph.add_edge("approval", "finalize")
graph.set_finish_point("finalize")

app = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-1"}}
app.invoke(
    {
        "request": "Explain the intuition behind SHAP values for model interpretability",
        "draft": "",
        "final": "",
    },
    config,
)

state = app.get_state(config)
print(f"Next node: {state.next}\nDraft preview: {state.values['draft'][:200]}...")

feedback = "Make it more concise and add a concrete example with a binary classifier."
result = app.invoke(Command(resume={"approved": False, "feedback": feedback}), config)
print(f"Final ({len(result['final'].split())} words):\n{result['final'][:300]}...")

Next node: ('approval',)
Draft preview: **Intuition Behind SHAP Values**

SHAP (SHapley Additive exPlanations) answers one question: *How much did each feature contribute to this specific prediction, compared to the average prediction?*

**...
Final (299 words):
Here’s a revised version of the explanation, made more concise and updated with a concrete binary classifier example.

---

**Intuition Behind SHAP Values**

SHAP (SHapley Additive exPlanations) answers: *How much did each feature contribute to this specific prediction, compared to the average?*

**...


In [12]:
# 11. PERSISTENT MULTI-TURN CHAT (IN-MEMORY)
# A stateful conversation graph that remembers prior messages


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


def chat(state: ChatState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


graph = StateGraph(ChatState)
graph.add_node("chat", chat)
graph.set_entry_point("chat")
graph.set_finish_point("chat")

app = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "ml-session-1"}}

result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="I'm studying gradient descent. Can you give me a quick overview?"
            )
        ]
    },
    config,
)
print("Turn 1:", result["messages"][-1].content)

result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="What's the difference between batch and stochastic variants?"
            )
        ]
    },
    config,
)
print("Turn 2:", result["messages"][-1].content)

state = app.get_state(config)
print(f"Total messages in state: {len(state.values['messages'])}")

Turn 1: This is a perfect topic. Here is a quick, intuitive overview of **Gradient Descent**.

### The Core Idea

Imagine you are blindfolded on a mountain and want to get to the valley floor (the lowest point). You can't see the whole path, but you can feel the ground beneath your feet.

- **Gradient:** The "steepness" and direction of the slope right where you are standing.
- **Descent:** You take a step in the *opposite* direction of the steepest slope (downhill).

You repeat this process: feel the slope, step downhill, feel the slope again, step again. Eventually, you reach the bottom.

### In Machine Learning Terms

1.  **The Mountain:** This is your **Loss Function** (e.g., Mean Squared Error). It measures how wrong your model's predictions are. You want to make this number as small as possible.
2.  **Your Position:** This is your model's current **Parameters** (e.g., the weights `w` and bias `b` in a linear regression).
3.  **The Slope (Gradient):** This is a mathematical calcul

In [18]:
# 12. PERSISTENT CHAT WITH POSTGRES (CROSS-SESSION MEMORY)
# Swap InMemorySaver for PostgresSaver to persist state across processes

uri = os.environ["POSTGRES_URI"]
config = {"configurable": {"thread_id": "persistent-ml-1"}}

with PostgresSaver.from_conn_string(uri) as saver:
    saver.setup()
    app = graph.compile(saver)
    app.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Key insight: L2 regularization is equivalent to a Gaussian prior on weights."
                )
            ]
        },
        config,
    )

with PostgresSaver.from_conn_string(uri) as saver:
    app = graph.compile(saver)
    result = app.invoke(
        {
            "messages": [
                HumanMessage(content="What regularization insight did we discuss?")
            ]
        },
        config,
    )
    print("Session 2:", result["messages"][-1].content)

Session 2: Based on our conversation, the key regularization insight we discussed is:

**L2 regularization is mathematically equivalent to assuming a Gaussian (Normal) prior distribution on the model's weights in a Bayesian framework.**

To be more specific, the insight is that performing **Maximum a Posteriori (MAP) estimation** with a Gaussian prior on the weights yields the exact same loss function as minimizing the sum of squared errors with an L2 penalty (Ridge Regression).

### The Core Takeaway

This insight reframes regularization from a "penalty trick" to a **principled Bayesian belief**:

- **Frequentist View:** You add $\lambda \sum w_j^2$ to the loss function to shrink weights and prevent overfitting.
- **Bayesian View:** You start with the belief that weights are likely small and centered around zero (a Gaussian prior $w_j \sim \mathcal{N}(0, \tau^2)$). The data then updates this belief.

The regularization parameter $\lambda$ is no longer arbitrary; it is the ratio of the

In [19]:
# 13. STATE INSPECTION AND TIME TRAVEL
# Introspect checkpoints and rewind to a previous node for branching


class AnalysisState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    step: str


def analyze(state: AnalysisState) -> dict:
    return {"messages": [llm.invoke(state["messages"])], "step": "analyzed"}


def summarize(state: AnalysisState) -> dict:
    last = state["messages"][-1]
    text = last.content if isinstance(last.content, str) else str(last.content)
    return {
        "messages": [
            llm.invoke(
                [
                    HumanMessage(
                        content=f"Summarize in one sentence for a practitioner: {text}"
                    )
                ]
            )
        ],
        "step": "summarized",
    }


graph = StateGraph(AnalysisState)
graph.add_node("analyze", analyze)
graph.add_node("summarize", summarize)
graph.set_entry_point("analyze")
graph.add_edge("analyze", "summarize")
graph.set_finish_point("summarize")

app = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "inspect-demo"}}

app.invoke(
    {
        "messages": [HumanMessage(content="Explain the curse of dimensionality")],
        "step": "",
    },
    config,
)

state = app.get_state(config)
print(f"step: {state.values['step']}, next: {state.next or '(finished)'}")
print(f"checkpoint_id: {state.config['configurable']['checkpoint_id']}")

for i, snapshot in enumerate(app.get_state_history(config)):
    writes = snapshot.metadata.get("writes") or {}
    node = next(iter(writes), "input")
    print(
        f"[{i}] node={node:<14} step={snapshot.metadata.get('step', '?')} messages={len(snapshot.values.get('messages', []))} next={snapshot.next or '()'}"
    )
    if i >= 3:
        break

step: summarized, next: (finished)
checkpoint_id: 1f170d2b-2d02-67a4-8002-52b1669a5683
[0] node=input          step=2 messages=3 next=()
[1] node=input          step=1 messages=2 next=('summarize',)
[2] node=input          step=0 messages=1 next=('analyze',)
[3] node=input          step=-1 messages=0 next=('__start__',)


In [20]:
# 14. BRANCHING FROM A CHECKPOINT
# Fork the same base state into two independent follow-up threads

app = graph.compile(InMemorySaver())
main_config = {"configurable": {"thread_id": "main"}}
app.invoke(
    {
        "messages": [
            HumanMessage(content="Summarize the key ideas behind ensemble learning.")
        ],
        "step": "",
    },
    main_config,
)

base_values = app.get_state(main_config).values

for thread_id, followup in [
    ("branch-bagging", "How does bagging reduce variance specifically?"),
    ("branch-boosting", "How does boosting reduce bias specifically?"),
]:
    branch_config = {"configurable": {"thread_id": thread_id}}
    app.update_state(branch_config, base_values)
    result = app.invoke({"messages": [HumanMessage(content=followup)]}, branch_config)
    print(f"{thread_id}: {result['messages'][-1].content[:200]}...")

branch-bagging: Bagging reduces variance by training models on different bootstrap samples (data perturbation) so their errors are decorrelated, then averaging their predictions to cancel out noise....
branch-boosting: Boosting reduces bias by sequentially training new models to correct the errors (residuals) of previous ones, forcing the ensemble to focus on and master the hardest-to-predict cases....
